In [13]:
import pandas as pd
import numpy as np
from utils import data as utils_data
from utils import tokenize as utils_tokenize
from umap import UMAP
import plotly.express as px
import hdbscan

In [2]:
df = utils_data.load_and_clean_data(sample=True)

/Users/hunain/d/coding-projects/biodiversity-taxonomy-thesis/utils/data.py:18: DtypeWarning: Columns (2,3,4,6,10,11,14,15,16,17,18,42,47,49,50,51,55,58,59,67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Sampling 1000 rows from the dataset of length 359435.
Dataset shape: (1000, 72)
 Count Summary : {'row_count': 1000, 'web_of_science_record_count': np.int64(1000), 'web_of_science_record_distinct_count': 1000, 'duplicated_row_count': np.int64(0)}
Dropped 0 duplicated rows. Remaining rows: 1000
 6 Columns with null values:
Highly Cited Status    992
Hot Paper Status       992
Author Keywords        193
Keywords Plus          134
Publication Date       133
Abstract                41
dtype: int64
Dropped 41 rows without Abstract. Remaining rows: 959


In [3]:
tokenized_data= utils_tokenize.tokenize_dataframe_fields(df, model_name="bert-base-uncased", max_length=512)


📦 Loading tokenizer: bert-base-uncased
🧹 Cleaning and combining fields...
🔠 Tokenizing individual fields...
🔗 Tokenizing combined field...

📏 Token shapes:
Title: torch.Size([959, 71])
Abstract: torch.Size([959, 512])
Author Keywords: torch.Size([959, 92])
Keywords Plus: torch.Size([959, 77])
Research Areas: torch.Size([959, 20])
Combined: torch.Size([959, 512])

🔍 Sample tokenized title input_ids:
tensor([  101,  4439, 10595,  1997,  5800, 17341,  1155,  1998,  1006,  1017,
         8906,  2104,  2146,  1011,  2744, 14114,  2804,  1024,  4942,  6494,
        15277, 21770, 10624,  6914,  3989,  2241,  2006,  1996,  4044,  4989,
          102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0])


In [4]:
embeddings, text_df = utils_tokenize.encode_tokenized_inputs_with_text(tokenized_data, field="combined", model_name="bert-base-uncased")


📤 Encoding field: 'combined' using model: bert-base-uncased
⚙️  Using device: mps


In [ ]:
embeddings = embeddings.cpu().numpy()
embeddings.shape 

torch.Size([959, 768])

In [12]:
import plotly.graph_objects as go

print("🔍 Running UMAP...")
umap_model = UMAP(n_components=2, random_state=42, min_dist=0.1,metric="cosine", n_neighbors=5)
umap_coords = umap_model.fit_transform(embeddings)

# Prepare DataFrame for plotting
plot_df = pd.DataFrame(umap_coords, columns=[f"UMAP_{i+1}" for i in range(2)])
plot_df['text'] = text_df['combined_text']

# Plot
def wrap_text(text, width=80):
    return "<br>".join([text[i:i+width] for i in range(0, len(text), width)])

plot_df['wrapped_text'] = plot_df['text'].apply(wrap_text)

fig = go.Figure(
    data=go.Scattergl(
        x=plot_df["UMAP_1"],
        y=plot_df["UMAP_2"],
        mode='markers',
        marker=dict(size=6, opacity=0.7),
        text=plot_df['wrapped_text'],
        hovertemplate='%{text}<extra></extra>'
    )
)

fig.update_layout(
    title='UMAP Projection of Embeddings',
    height=650,
    margin=dict(l=10, r=10, t=40, b=10)
)

fig.show()

🔍 Running UMAP...


/Users/hunain/d/coding-projects/biodiversity-taxonomy-thesis/env/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:
def cluster_and_plot_hdbscan(umap_embeddings, text_df, text_column='combined_text', min_cluster_size=5):
    """
    Runs HDBSCAN clustering and plots UMAP points with interactive tooltips.

    Args:
        umap_embeddings (np.ndarray): 2D UMAP coordinates (N, 2).
        text_df (pd.DataFrame): Original text for hover info.
        text_column (str): Column to use in hover tooltips.
        min_cluster_size (int): Minimum size of clusters in HDBSCAN.

    Returns:
        Tuple: (plotly figure, cluster labels)
    """
    print("🔍 Running HDBSCAN...")
    clusterer = hdbscan.HDBSCAN( 
        cluster_selection_method = "leaf",
        metric="euclidean",
        min_samples=1,  # Set to 1 to allow single points as clusters
        )
    print(clusterer.labels_)
    labels = clusterer.fit_predict(umap_embeddings)

    # Prepare DataFrame for plotting
    plot_df = pd.DataFrame(umap_embeddings, columns=["UMAP_1", "UMAP_2"])
    plot_df['cluster'] = labels
    plot_df['text'] = text_df[text_column].astype(str)

    # Wrap text for better tooltip display
    def wrap_text(text, width=80):
        return "<br>".join([text[i:i+width] for i in range(0, len(text), width)])
    plot_df['wrapped_text'] = plot_df['text'].apply(wrap_text)

    print(f"📊 Found {len(set(labels)) - (1 if -1 in labels else 0)} clusters, with {sum(labels == -1)} noise points")

    fig = px.scatter(
        plot_df,
        x="UMAP_1",
        y="UMAP_2",
        color=plot_df['cluster'].astype(str),
        hover_data={'cluster': True, 'UMAP_1': False, 'UMAP_2': False, 'wrapped_text': True},
        title="UMAP + HDBSCAN Clustering"
    )

    fig.update_traces(marker=dict(size=6, opacity=0.7))
    fig.update_layout(height=650)
    fig.show()

    return fig, labels

In [20]:
# embeddings, text_df = encode_tokenized_inputs_with_text(...)
# umap_embeddings = UMAP(...).fit_transform(embeddings.numpy())
fig, labels = cluster_and_plot_hdbscan(umap_coords, text_df, text_column="combined_text")

🔍 Running HDBSCAN...


AttributeError: 'HDBSCAN' object has no attribute 'labels_'

In [18]:
clusterer.labels_

NameError: name 'clusterer' is not defined